In [ ]:
# ============================================================
# Nystrom.ipynb  
# ============================================================

import os
import platform
import json
import random
from fractions import Fraction
import pickle
import gc
import time

import numpy as np
import torch
torch.set_num_threads(16)

import torch.nn as nn
import torch.optim as optim

from sklearn import svm
from sklearn.svm import SVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

from data_loader import *
from common_functions import *
import interferometer as itf  # kept for environment consistency


# -----------------------
# Global device
# -----------------------
device = torch.device("cpu")

# ============================================================
# Photonic kernel between two sets
#   - TRAIN version: grad-enabled
#   - EVAL version: no_grad wrapper
# ============================================================
def _kernel_between_sets_photonic_core(
    XA, XB,
    modes: int,
    depth: int,
    init_state,
    block_a: int = 64,
    block_b: int = 64
):
    """
    K_ab = |Per(U(a)^† U(b)[idx,idx])|^2
    """
    XA = torch.stack(XA) if isinstance(XA, list) else XA
    XB = torch.stack(XB) if isinstance(XB, list) else XB
    XA = XA.to(device)
    XB = XB.to(device)

    NA = XA.shape[0]
    NB = XB.shape[0]

    def build_U(X):
        N = X.shape[0]
        TBUargs = X.view(N, -1, 2)
        theta = (TBUargs[:, :, 0] * (torch.pi / 2)).to(torch.float32)
        phi   = (TBUargs[:, :, 1] * (2 * torch.pi)).to(torch.float32)

        cos_theta = torch.cos(theta).to(torch.cfloat)
        sin_theta = torch.sin(theta).to(torch.cfloat)
        exp_phi   = torch.exp(1j * phi.to(torch.cfloat))

        num_TBU = TBUargs.size(1)
        TBUunitaries = torch.zeros((N, num_TBU, 2, 2), dtype=torch.cfloat, device=device)
        TBUunitaries[:, :, 0, 0] = exp_phi * cos_theta
        TBUunitaries[:, :, 0, 1] = -sin_theta
        TBUunitaries[:, :, 1, 0] = exp_phi * sin_theta
        TBUunitaries[:, :, 1, 1] = cos_theta

        U = torch.eye(modes, dtype=torch.cfloat, device=device).repeat(N, 1, 1)
        N_before = 0
        for d in range(depth):
            U, N_before = apply_depth_layer_batch(d, modes, TBUunitaries, U, N_before)
        return U

    UA = build_U(XA)
    UB = build_U(XB)

    idx = torch.tensor([mode for mode, count in enumerate(init_state) for _ in range(count)],
                       dtype=torch.long, device=device)

    K = torch.zeros((NA, NB), dtype=torch.float32, device=device)

    for a0 in range(0, NA, block_a):
        a1 = min(a0 + block_a, NA)
        UA_blk = UA[a0:a1]
        UA_blk_dag = UA_blk.conj().transpose(-1, -2)

        for b0 in range(0, NB, block_b):
            b1 = min(b0 + block_b, NB)
            UB_blk = UB[b0:b1]

            Uab = torch.matmul(UA_blk_dag.unsqueeze(1), UB_blk.unsqueeze(0))
            Uab_sub = Uab[:, :, idx[:, None], idx[None, :]]

            ba = Uab_sub.shape[0]
            bb = Uab_sub.shape[1]
            nph = Uab_sub.shape[2]

            Uflat = Uab_sub.reshape(ba * bb, nph, nph)

            perm = permanent_ryser(Uflat)  # complex
            P = (torch.abs(perm) ** 2).to(torch.float32)

            K[a0:a1, b0:b1] = P.reshape(ba, bb)

    return K

def kernel_between_sets_photonic_train(*args, **kwargs):
    return _kernel_between_sets_photonic_core(*args, **kwargs)

@torch.no_grad()
def kernel_between_sets_photonic_eval(*args, **kwargs):
    return _kernel_between_sets_photonic_core(*args, **kwargs)


# ============================================================
# PSD pseudoinverse (same for TRAIN and EVAL)
# ============================================================
def psd_pinv_eigh(W: torch.Tensor, eps_rel: float = 1e-10) -> torch.Tensor:
    """
    W: symmetric (m,m), float64 recommended.
    returns: W^+ (m,m)
    """
    Wsym = 0.5 * (W + W.T)
    evals, evecs = torch.linalg.eigh(Wsym)

    lam_max = torch.clamp(evals.max(), min=1e-12)
    eps = eps_rel * lam_max                  # 0-d tensor is fine
    evals_clamped = torch.clamp(evals, min=eps)  # <-- FIX

    Winv = (evecs * (1.0 / evals_clamped)) @ evecs.T
    # debug
    # rank = int((evals > eps).sum().item())
    # print("clamped =", W.shape[0] - rank)
    # print(f"[pinv] m={W.shape[0]} lam_max={lam_max.item():.3e} eps={eps.item():.3e} rank={rank}")
    return 0.5 * (Winv + Winv.T)


# ============================================================
# Nyström kernels
# ============================================================
def nystrom_train_kernel_torch(
    X_train_emb: torch.Tensor,   # (Ntr, d)
    landmark_idx: torch.Tensor,  # (m,)
    modes: int,
    depth: int,
    init_state,
    block_a: int = 64,
    block_b: int = 64,
    eps_rel: float = 1e-10,
):
    X_land = X_train_emb[landmark_idx]

    # compute W and C in float64 for stable eig
    W = kernel_between_sets_photonic_train(
        X_land, X_land, modes, depth, init_state, block_a, block_b
    ).to(dtype=torch.float64)
    Ctr = kernel_between_sets_photonic_train(
        X_train_emb, X_land, modes, depth, init_state, block_a, block_b
    ).to(dtype=torch.float64)

    W_pinv = psd_pinv_eigh(W, eps_rel=eps_rel)

    Khat = Ctr @ W_pinv @ Ctr.T
    Khat = 0.5 * (Khat + Khat.T)
    Khat = Khat.to(dtype=torch.float32)
    Khat.fill_diagonal_(1.0)
    return Khat


@torch.no_grad()
def nystrom_eval_kernels_numpy(
    X_train_emb: torch.Tensor,   # (Ntr, d)
    X_test_emb: torch.Tensor,    # (Nte, d)
    landmark_idx: torch.Tensor,  # (m,)
    modes: int,
    depth: int,
    init_state,
    block_a: int = 64,
    block_b: int = 64,
    eps_rel: float = 1e-10,
):
    X_land = X_train_emb[landmark_idx]

    W = kernel_between_sets_photonic_eval(
        X_land, X_land, modes, depth, init_state, block_a, block_b
    ).to(dtype=torch.float64)
    Ctr = kernel_between_sets_photonic_eval(
        X_train_emb, X_land, modes, depth, init_state, block_a, block_b
    ).to(dtype=torch.float64)
    Cte = kernel_between_sets_photonic_eval(
        X_test_emb, X_land, modes, depth, init_state, block_a, block_b
    ).to(dtype=torch.float64)

    W_pinv = psd_pinv_eigh(W, eps_rel=eps_rel)

    Ktr = (Ctr @ W_pinv @ Ctr.T).to(dtype=torch.float32)
    Ktr = 0.5 * (Ktr + Ktr.T)
    Ktr.fill_diagonal_(1.0)

    Kte = (Cte @ W_pinv @ Ctr.T).to(dtype=torch.float32)

    # Keep within [0,1] lightly (probability kernel); optional but safe
    Ktr = torch.clamp(Ktr, 0.0, 1.0)
    Ktr.fill_diagonal_(1.0)
    Kte = torch.clamp(Kte, 0.0, 1.0)

    return Ktr.cpu().numpy(), Kte.cpu().numpy()


# ============================================================
# QuantumKernelNN
# ============================================================
class QuantumKernelNN(nn.Module):
    def __init__(self, input_dim, output_dim, init_state, modes, depth):
        super().__init__()
        self.device = device
        self.fc1 = nn.Linear(input_dim, output_dim).to(self.device)
        self.init_state = init_state
        self.modes = modes
        self.depth = depth

    def forward(self, x, return_kernel: bool = False):
        x = x.to(self.device)
        x_emb = torch.sigmoid(self.fc1(x))  # (N, num_circ_params)
        if return_kernel:
            raise RuntimeError("This script trains/evals using Nyström only; set return_kernel=False.")
        return x_emb, None


# ============================================================
# Main experiment: train per m using Nyström loss
# ============================================================
def run_nystrom_sweep_bsnn_aligned(
    N,
    test_portion,
    num_epochs,
    n_iterations,
    data_type,
    folder_path,
    modes,
    depth,
    num_photons,
    m_portions,
    block_a=64,
    block_b=64,
    eps_rel_pinv: float = 1e-10,
    SEED=135):
    os.makedirs(folder_path, exist_ok=True)

    X_data, y_data, X_train_all, X_test_all, y_train_all, y_test_all = prepare_data(data_type)

    is_mnist_like = data_type in (
        'cifar10','MNIST','MNISTresized','fashionMNIST','MNIST01','MNIST17',
        'fashionMNIST_Tshirt_dress','organMNISTa','organMNISTb'
    )

    for jj in range(n_iterations):
        seed_model = int(SEED) + int(jj)
        set_all_seeds(seed_model)
        rng_state = get_rng_state()

        # deterministic split for this jj
        if is_mnist_like:
            idx_tr, idx_te = get_or_make_mnist_sep_indices(
                folder_path=folder_path,
                data_type=data_type,
                N=N,
                test_portion=test_portion,
                jj=jj,
                n_train_total=len(X_train_all),
                n_test_total=len(X_test_all),
                seed_base=SEED
            )
            (X_raw, X_raw_train, X_raw_test, y_labels, y_train, y_test,
             subset_indexes_train, subset_indexes_test, global_indexes_train, global_indexes_test) = \
                build_from_mnist_separate(X_train_all, y_train_all, X_test_all, y_test_all, idx_tr, idx_te)
        else:
            total_len = len(X_data)
            idx_subset, idx_train_global, idx_test_global = get_or_make_split_indices(
                folder_path=folder_path,
                data_type=data_type,
                N=N,
                test_portion=test_portion,
                jj=jj,
                total_len=total_len,
                seed_base=SEED
            )
            (X_raw, X_raw_train, X_raw_test, y_labels, y_train, y_test,
             subset_indexes_train, subset_indexes_test, global_indexes_train, global_indexes_test) = \
                build_from_global_split(X_data, y_data, idx_subset, idx_train_global, idx_test_global)

        if num_photons > modes:
            print("Skipping: num_photons > modes")
            continue

        state_name, init_state = get_leftmost_state(modes, num_photons)

        # z_ij target (train×train)
        num_train = int((1 - test_portion) * N)
        z_ij = zij_calc(num_train, y_train).to(device=device, dtype=torch.float32)

        _, num_circ_params = pad_input(X_raw, modes, depth, init_state)

        # tensors for model input
        X_train_tensor = torch.stack([x.clone().detach().to(device) for x in X_raw_train])
        X_test_tensor  = torch.stack([x.clone().detach().to(device) for x in X_raw_test])

        y_train_np = y_train.cpu().numpy() if torch.is_tensor(y_train) else np.array(y_train)
        y_test_np  = y_test.cpu().numpy()  if torch.is_tensor(y_test)  else np.array(y_test)

        N_train_eff = len(X_raw_train)

        # compute m_list from portions of N_train
        m_list = []
        for p in m_portions:
            p = float(p)
            if p <= 0:
                continue
            m = int(round(p * N_train_eff))
            m = max(1, min(m, N_train_eff))
            m_list.append(m)
        m_list = sorted(set(m_list), reverse=False)

        for m in m_list:
            # --------------------------------------------
            # SKIP if this (jj,m) result already exists
            # --------------------------------------------
            save_path = os.path.join(
                folder_path,
                f"nystrom_train_{data_type}_N{N}_phot{num_photons}_modes{modes}_depth{depth}_jj{jj}_m{m}.json"
            )

            if os.path.exists(save_path):
                try:
                    with open(save_path, "r") as f:
                        old = json.load(f)
                    has_final = ("final" in old) and ("acc_test" in old["final"])
                    has_epochs = ("epochs" in old) and (len(old["epochs"]) >= int(num_epochs))
                    if has_final and has_epochs:
                        print(f"[skip] exists and complete: jj={jj} m={m} -> {save_path}")
                        continue
                    else:
                        print(f"[redo] exists but incomplete: jj={jj} m={m} -> {save_path}")
                except Exception as e:
                    print(f"[redo] exists but unreadable ({e}): jj={jj} m={m} -> {save_path}")

            # BSNN: reset rng BEFORE model init and advance
            set_rng_state(rng_state)
            _ = torch.rand(3)
            _ = np.random.rand(3)

            nn_in_dim = len(X_raw[0])
            nn_out_dim = num_circ_params

            model = QuantumKernelNN(
                input_dim=nn_in_dim,
                output_dim=nn_out_dim,
                init_state=init_state,
                modes=modes,
                depth=depth
            ).to(device)

            optimizer = optim.Adam(model.parameters(), lr=0.001)
            criterion = nn.MSELoss()

            # ------------------------------------------------------------
            # FIXED landmarks for this (jj, m)
            # ------------------------------------------------------------
            if int(m) == int(N_train_eff):
                landmark_idx = torch.arange(N_train_eff, device=device, dtype=torch.long)
                seed_land = None
            else:
                seed_land = (SEED + jj) * 1_000_000 + int(m)
                rng_land = np.random.RandomState(seed_land)
                landmark_idx_np = rng_land.choice(N_train_eff, size=int(m), replace=False)
                landmark_idx = torch.tensor(landmark_idx_np, dtype=torch.long, device=device)

            out = {
                "data_type": data_type,
                "N": int(N),
                "test_portion": float(test_portion),
                "N_train": int(N_train_eff),
                "N_test": int(len(X_raw_test)),
                "modes": int(modes),
                "depth": int(depth),
                "num_photons": int(num_photons),
                "state_name": str(state_name),
                "jj": int(jj),
                "m": int(m),
                "m_fraction_of_train": float(m / max(1, N_train_eff)),
                "SEED": int(SEED),
                "seed_landmarks": (None if seed_land is None else int(seed_land)),
                "epochs": [],
                "final": {}
            }

            # -----------------------
            # Train for this m
            # -----------------------
            for epoch in range(num_epochs):
                optimizer.zero_grad()

                X_prime_train, _ = model(X_train_tensor, return_kernel=False)

                Khat_train = nystrom_train_kernel_torch(
                    X_train_emb=X_prime_train,
                    landmark_idx=landmark_idx,
                    modes=modes,
                    depth=depth,
                    init_state=init_state,
                    block_a=block_a,
                    block_b=block_b,
                    eps_rel=eps_rel_pinv
                )

                loss = criterion(Khat_train, z_ij)
                loss.backward()
                optimizer.step()

                if epoch % 10 == 0:
                    print(f"[jj={jj} m={m} epoch={epoch+1}] nystrom-loss={loss.item():.6f}")

                out["epochs"].append({"epoch": int(epoch), "loss": float(loss.item())})

            # -----------------------
            # Final accuracy for this m (after training)
            # -----------------------
            with torch.no_grad():
                X_prime_train, _ = model(X_train_tensor, return_kernel=False)
                X_prime_test,  _ = model(X_test_tensor,  return_kernel=False)

                Ktr_hat, Kte_hat = nystrom_eval_kernels_numpy(
                    X_train_emb=X_prime_train,
                    X_test_emb=X_prime_test,
                    landmark_idx=landmark_idx,
                    modes=modes,
                    depth=depth,
                    init_state=init_state,
                    block_a=block_a,
                    block_b=block_b,
                    eps_rel=eps_rel_pinv
                )

                num_classes = len(np.unique(y_train_np))
                if num_classes > 2:
                    clf = OneVsRestClassifier(SVC(kernel="precomputed"))
                else:
                    clf = svm.SVC(kernel="precomputed")

                clf.fit(Ktr_hat, y_train_np)
                y_pred = clf.predict(Kte_hat)
                acc = accuracy_score(y_test_np, y_pred)

            out["final"] = {"acc_test": float(acc)}
            print(f"[jj={jj} m={m}] FINAL acc={acc:.4f}")

            save_path = os.path.join(
                folder_path,
                f"nystrom_train_{data_type}_N{N}_phot{num_photons}_modes{modes}_depth{depth}_jj{jj}_m{m}.json"
            )
            with open(save_path, "w") as f:
                json.dump(out, f, indent=2)





data_type = "ionosphere"
X_data, y_data, X_train, X_test, y_train, y_test = prepare_data(data_type)

if data_type in ['MNIST','fashionMNIST']:
    N = 5000
    SEED = 135
    test_portion = 1 / 7
    modes = 12
else:
    SEED = 110
    N = len(X_data)
    test_portion = 0.2

modes = 6

portion = Fraction(test_portion).limit_denominator()
numerator = portion.numerator
denominator = portion.denominator

if platform.system() == "Windows":
    base_path = r"E:\BSNN"
else:
    home = os.path.expanduser("~")
    base_path = os.path.join(home, "BSNN")

folder_path = os.path.join(base_path, f"{data_type}_test_portion_{numerator}_of_{denominator}_NYSTROM")
os.makedirs(folder_path, exist_ok=True)


depth = modes


factor = 1- test_portion
m_portions = [(x / N) / factor for x in list(range(1, 20, 2)) + list(range(20, 100, 10))]
num_epochs = 50
n_iterations = 10

block_a = 32
block_b = 32

for num_photons in range(1,6,1):
    run_nystrom_sweep_bsnn_aligned(
        N=N,
        test_portion=test_portion,
        num_epochs=num_epochs,
        n_iterations=n_iterations,
        data_type=data_type,
        folder_path=folder_path,
        modes=modes,
        depth=depth,
        num_photons=num_photons,
        m_portions=m_portions,
        block_a=block_a,
        block_b=block_b,
        eps_rel_pinv=1e-10,         
        SEED=SEED    )